# 👄 VIDEO-FREE · 4. Que los labios sigan la canción (LatentSync)
[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ivanherrerabaquero-dev/VIDEO-FREE/blob/main/colab/4_labios_ia.ipynb)

Toma una escena de video donde se ve la **cara del cantante de frente** (por ejemplo una toma del cuaderno 3 hecha desde `actor_canta.png`)
y le mueve la boca para que **cante exactamente la parte de la canción** que suena en esa escena.

| GPU | Versión | Resolución de la cara |
|---|---|---|
| L4 / A100 (≥ 18 GB) | LatentSync 1.6 | 512 px |
| T4 gratis | LatentSync 1.5 | 256 px |

- **Licencias:** código Apache 2.0; modelos OpenRAIL++ (uso libre con prohibiciones de usos dañinos, como suplantar a otras personas).
- Funciona mejor con caras grandes, de frente y bien iluminadas. Si el video es más corto que el audio, lo repite de ida y vuelta.

**Antes de empezar (una sola vez):**
1. *Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU.*
2. Crea un token de GitHub (*Settings → Developer settings → Fine-grained tokens*) con acceso solo a **VIDEO-FREE** y permiso **Contents: Read and write**.
3. En Colab, abre el icono de **llave (Secretos)**, crea `GITHUB_TOKEN`, pega el token y activa *Acceso al cuaderno*.



In [ ]:
#@title 1 · Preparar Colab y conectar con tu repo
REPO = "ivanherrerabaquero-dev/VIDEO-FREE"  #@param {type:"string"}
RAMA = "main"  #@param {type:"string"}
import os, sys, subprocess
if os.path.exists("/content/VIDEO-FREE/.git"):
    subprocess.run(["git", "-C", "/content/VIDEO-FREE", "pull", "-q"], check=False)
else:
    subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", RAMA, f"https://github.com/{REPO}", "/content/VIDEO-FREE"], check=True)
sys.path.insert(0, "/content/VIDEO-FREE/colab")
import importlib, comun; importlib.reload(comun)
info = comun.preparar(REPO, RAMA)
VRAM = info["gpu"].get("vram_gb", 0)

In [ ]:
#@title 2 · Instalar LatentSync en un entorno aparte (5–10 min la primera vez)
import os
if not os.path.exists("/content/latentsync"):
    !git clone -q https://github.com/bytedance/LatentSync /content/latentsync
    !git -C /content/latentsync checkout -q a229c3948406bc2cf6eaf4873e662e70c6a04746
!pip install -q uv
!uv venv -q -p 3.10 /content/ls_env
!cd /content/latentsync && uv pip install -q -p /content/ls_env/bin/python -r requirements.txt
VERSION = "1.6" if VRAM >= 18 else "1.5"
CONFIG = "configs/unet/stage2_512.yaml" if VERSION == "1.6" else "configs/unet/stage2.yaml"
from huggingface_hub import hf_hub_download
for f in ["whisper/tiny.pt", "latentsync_unet.pt"]:
    hf_hub_download(f"ByteDance/LatentSync-{VERSION}", f, local_dir="/content/latentsync/checkpoints")
print("LatentSync", VERSION, "listo")

In [ ]:
#@title 3 · Elegir la escena
PROYECTO = "carretera-sin-ti.json"  #@param {type:"string"}
ESCENA = 3  #@param {type:"integer"}
#@markdown Número de la escena tal como aparece en la pestaña Escenas del portal, contando desde 1. Debe ser de tipo `video`.
USAR_VOZ_SOLA = True  #@param {type:"boolean"}
#@markdown Con la voz sola (sin instrumentos) la boca se sincroniza mejor.
PASOS = 20  #@param {type:"slider", min:10, max:50, step:5}

from pathlib import Path
p = comun.proyecto(PROYECTO)
e = p["escenas"][ESCENA - 1]
if e["tipo"] != "video" or not e.get("archivo"):
    raise SystemExit("Esa escena no es un video. Genera primero una toma con el cuaderno 3.")
clip = comun.bajar(e["archivo"])
audio_fuente = None
ia = p.get("ia") or {}
if USAR_VOZ_SOLA:
    camino_voz = (ia.get("cancion") or {}).get("voz")
    audio_fuente = comun.bajar(camino_voz) if camino_voz else comun.separar_voz(comun.bajar(p["audio"]))[0]
else:
    audio_fuente = comun.bajar(p["audio"])
tramo = comun.TRABAJO / f"tramo_{ESCENA}.wav"
comun.sh(f'ffmpeg -hide_banner -loglevel error -y -ss {e["desde"]} -to {e["hasta"]} -i "{audio_fuente}" -ac 1 -ar 16000 "{tramo}"')
print(f"Escena {ESCENA}: {e['desde']}–{e['hasta']} s · clip {e['archivo']}"); comun.escuchar(tramo)

In [ ]:
#@title 4 · Sincronizar los labios
salida = comun.TRABAJO / f"labios_{ESCENA}.mp4"
comun.sh(f'/content/ls_env/bin/python -m scripts.inference --unet_config_path "{CONFIG}" --inference_ckpt_path checkpoints/latentsync_unet.pt '
         f'--inference_steps {PASOS} --guidance_scale 1.5 --enable_deepcache --video_path "{clip}" --audio_path "{tramo}" --video_out_path "{salida}"',
         cwd="/content/latentsync")
final = comun.TRABAJO / f"labios_{ESCENA}_h264.mp4"
comun.sh(f'ffmpeg -hide_banner -loglevel error -y -i "{salida}" -c:v libx264 -crf 20 -pix_fmt yuv420p -movflags +faststart -an "{final}"')
comun.ver(salida)

In [ ]:
#@title 5 · Guardar en tu repo y usarla en la escena
nombre = comun.slug(p.get("titulo", "video"))
destino = f"medios/escenas/{nombre}_labios_escena{ESCENA}.mp4"
comun.a_drive(final, Path(destino).name)
comun.guardar(destino, final.read_bytes(), f"Labios sincronizados: escena {ESCENA}")
e["ia"] = {**(e.get("ia") or {}), "sinLabios": e["archivo"], "labios": f"LatentSync {VERSION}"}
e["archivo"], e["inicioClip"] = destino, 0
comun.guardar_proyecto(PROYECTO, p, f"Proyecto: labios en la escena {ESCENA}")
print("Listo. En el portal pulsa «Recargar proyecto».")